*scenario 1: a single data scientist participating in an ML competition*

MLflow setup:
* Tracking server: no
* Backend store: local filesystem
* Artifact store: local filesystem

The experiments can be explored locally by launching the MLflow UI

In [1]:
import mlflow

In [ ]:
print(f"tracking URI:'{mlflow.get_tracking_uri()}' ") 
# use local folder mlrun as the default unless we specify otherwise;
#  the folder isn't necessarily available/created at this moment but will be auto-created throughout interaction

tracking URI:'file:///workspaces/mlops-zoomcamp/02-experiment_tracking/running-mlflow-examples/mlruns' 


In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
experiments = client.search_experiments()
for exp in experiments:
    print(exp.name, exp.experiment_id) # at this interaction folder mlrun is created automatically
    # whenever you start using mlflow there will be a default experiment with id = 0

Default 0


In [7]:
# code for model training
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from mlflow.models import infer_signature

mlflow.set_experiment("my-experiment-1") # setting an experiment name that doesn't exist bofore; mlflow will create it automatically

with mlflow.start_run() as run:
    X, y = load_iris(return_X_y=True)

    params = {"C":0.1, "random_state":42}
    mlflow.log_params(params)

    lr= LogisticRegression(**params).fit(X,y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

# Define input example and signature before logging the model
    input_example = X[0:1] # keeps shape (1, n_features)
    signature = infer_signature(X, y_pred) # infer_signature(X, model.predict(X))

    mlflow.sklearn.log_model(lr, name="models", registered_model_name="models", input_example=input_example, signature=signature)
    print("Run_ID:", run.info.run_id)
    print("Artifact URI:", run.info.artifact_uri)
    print(f"default artifacts URI: '{mlflow.get_tracking_uri()}' ")


Run_ID: fef6f5e7acb44c52aa2919500f496bed
Artifact URI: file:///workspaces/mlops-zoomcamp/02-experiment_tracking/running-mlflow-examples/mlruns/580193361566889052/fef6f5e7acb44c52aa2919500f496bed/artifacts
default artifacts URI: 'file:///workspaces/mlops-zoomcamp/02-experiment_tracking/running-mlflow-examples/mlruns' 


Registered model 'models' already exists. Creating a new version of this model...
Created version '2' of model 'models'.


In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
experiments = client.search_experiments()
for exp in experiments:
    print(exp.name, exp.experiment_id)

    # the sub-folders within 580193361566889052 are for all submitted runs (successful or not)

my-experiment-1 580193361566889052
Default 0


Interacting with the model registry

In [3]:
from mlflow.tracking import MlflowClient
from mlflow.exceptions import MlflowException

client = MlflowClient()

try:
    models = client.search_registered_models() 
    for model in models:
        print(model.name)
except MlflowException as e :
    print(f"Mlflow-related error: {e}")
except Exception as e:
    print(f"General error occured: {e}")

models


to explore models locally using mlflow ui --> in the Terminal navigate to the folder where mlruns is saved (running-mlflow-exmaples) then use 'mlflow ui' command to activate the mlflow ui